PRIMER PUNTO

Código de consolidación y generación del dataset definitivo

In [ ]:
# @title
# ============================================================
# PROYECTO MACHINE LEARNING II
# CONSOLIDACIÓN Y GENERACIÓN DEL DATASET DEFINITIVO
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. CARGA DE LOS 4 DATASETS ORIGINALES
# ============================================================

X_train = pd.read_excel("X_train_balanced.xlsx")
y_train = pd.read_excel("y_train_balanced.xlsx")

X_test = pd.read_excel("X_test.xlsx")
y_test = pd.read_excel("y_test.xlsx")


# ============================================================
# 2. UNIÓN DE VARIABLES PREDICTORAS Y VARIABLE OBJETIVO
# ============================================================

train = pd.concat([X_train, y_train], axis=1)
test = pd.concat([X_test, y_test], axis=1)


# ============================================================
# 3. CONSOLIDACIÓN DE ENTRENAMIENTO Y PRUEBA
# ============================================================

df = pd.concat(
    [train, test],
    ignore_index=True
)

print("Dimensiones iniciales:", df.shape)

print("\nDistribución inicial de nivel_motivacion:")
print(df["nivel_motivacion"].value_counts())


# ============================================================
# 4. IDENTIFICACIÓN Y ELIMINACIÓN DE COLUMNAS DUPLICADAS
# ============================================================

duplicadas = df.columns[df.columns.duplicated()].tolist()

print("\nColumnas duplicadas:")
print(duplicadas)

df = df.loc[:, ~df.columns.duplicated()]

print("\nDimensiones después de eliminar duplicadas:", df.shape)


# ============================================================
# 5. IDENTIFICACIÓN DE VARIABLES POTENCIALMENTE RELACIONADAS
#    CON DATA LEAKAGE
# ============================================================

palabras_leakage = [
    "cancel",
    "aband",
    "abandon",
    "retir",
    "desert",
    "cambiarse",
    "cambiar",
    "carrera"
]

columnas_leakage = []

for col in df.columns:

    col_lower = col.lower()

    if any(
        palabra in col_lower
        for palabra in palabras_leakage
    ):
        columnas_leakage.append(col)


print("\nVariables potencialmente relacionadas con leakage:")

for col in columnas_leakage:
    print(col)


# ============================================================
# 6. ELIMINACIÓN DE VARIABLES SELECCIONADAS POR POSIBLE LEAKAGE
# ============================================================

# Variables identificadas durante la auditoría
columnas_excluir = [
    "razon_Dificultades académicas;",
    "razon_Dificultades académicas;Problemas financieros;",
    "razon_Falta de interés en la carrera o programa de estudios;",
    "razon_Falta de interés en la carrera o programa de estudios;Problemas personales o familiares;",
    "razon_Falta de interés por la carrera;",
    "razon_Problemas de salud mental;",
    "razon_Problemas de salud mental;Problemas personales o familiares;",
    "razon_Problemas financieros;",
    "razon_Problemas financieros;Dificultades académicas;",
    "razon_Problemas financieros;Problemas de salud mental;",
    "razon_Problemas financieros;Problemas personales o familiares;",
    "razon_Problemas financieros;Problemas personales o familiares;Problemas de salud mental;Dificultades académicas;",
    "razon_Problemas personales o familiares;",
    "razon_Problemas personales o familiares;Problemas de salud mental;",
    "razon_Problemas personales o familiares;Problemas financieros;",
    "r_carrera_Cambiarse de carrera;",
    "r_carrera_Cambiarse de carrera;Ninguno;",
    "r_carrera_Ninguno;"
]

# Mantener únicamente las columnas que realmente existan
columnas_excluir = [
    col for col in columnas_excluir
    if col in df.columns
]

df = df.drop(
    columns=columnas_excluir
)

print("\nDimensiones después de la limpieza:", df.shape)


# ============================================================
# 7. SEPARAR LOS REGISTROS REALES
# ============================================================

df_real = df.copy()

print("\nRegistros reales:", len(df_real))


# ============================================================
# 8. DEFINIR VARIABLES PREDICTORAS
# ============================================================

target = "nivel_motivacion"

predictoras = [
    col for col in df_real.columns
    if col != target
]

print("\nNúmero de variables predictoras:", len(predictoras))


# ============================================================
# 9. CREAR GRUPOS DE VARIABLES
# ============================================================

grupos = {}

for col in predictoras:

    partes = col.split("_")

    if len(partes) >= 2:
        grupo = "_".join(partes[:2])
    else:
        grupo = partes[0]

    if grupo not in grupos:
        grupos[grupo] = []

    grupos[grupo].append(col)


print("\nNúmero de grupos de variables:", len(grupos))


# ============================================================
# 10. GENERACIÓN DE REGISTROS SINTÉTICOS
# ============================================================

n_sinteticos = 1000

rng = np.random.default_rng(42)

sinteticos = []

firmas_generadas = set()


# ============================================================
# 11. CREAR FIRMAS DE LOS REGISTROS REALES
#    PARA EVITAR COPIAS EXACTAS
# ============================================================

firmas_reales = set(
    tuple(row)
    for row in df_real[predictoras].to_numpy()
)


# ============================================================
# 12. GENERAR LOS REGISTROS SINTÉTICOS
# ============================================================

while len(sinteticos) < n_sinteticos:

    # Seleccionar una categoría de nivel_motivacion
    categoria = rng.choice(
        df_real[target].unique()
    )

    # Filtrar estudiantes de la misma categoría
    candidatos = df_real[
        df_real[target] == categoria
    ]

    # Seleccionar dos perfiles diferentes
    padres = candidatos.sample(
        n=2,
        random_state=int(
            rng.integers(
                0,
                1_000_000
            )
        )
    )

    padre_1 = padres.iloc[0]
    padre_2 = padres.iloc[1]

    nueva_fila = {}


    # --------------------------------------------------------
    # Combinar grupos de variables de ambos perfiles
    # --------------------------------------------------------

    for grupo, columnas in grupos.items():

        if rng.random() < 0.5:
            padre = padre_1
        else:
            padre = padre_2

        for col in columnas:
            nueva_fila[col] = padre[col]


    # Mantener la categoría de riesgo
    nueva_fila[target] = categoria


    # --------------------------------------------------------
    # Crear firma del nuevo registro
    # --------------------------------------------------------

    firma = tuple(
        nueva_fila[col]
        for col in predictoras
    )


    # --------------------------------------------------------
    # Evitar copias exactas de registros reales
    # --------------------------------------------------------

    if firma in firmas_reales:
        continue


    # --------------------------------------------------------
    # Evitar duplicados entre sintéticos
    # --------------------------------------------------------

    if firma in firmas_generadas:
        continue


    firmas_generadas.add(firma)

    sinteticos.append(
        nueva_fila
    )


# Convertir a DataFrame
df_sintetico = pd.DataFrame(
    sinteticos
)

print("\nRegistros sintéticos generados:",
      len(df_sintetico))


# ============================================================
# 13. IDENTIFICAR LA FUENTE DE CADA REGISTRO
# ============================================================

df_real["FUENTE_DATO"] = "REAL"

df_sintetico["FUENTE_DATO"] = "SINTETICO"


# ============================================================
# 14. UNIR DATOS REALES Y SINTÉTICOS
# ============================================================

df_final = pd.concat(
    [
        df_real,
        df_sintetico
    ],
    ignore_index=True
)


# ============================================================
# 15. COMPROBACIONES FINALES
# ============================================================

print("\n======================================")
print("AUDITORÍA FINAL")
print("======================================")

print("\nDimensiones finales:")
print(df_final.shape)


print("\nFuente de los registros:")
print(
    df_final["FUENTE_DATO"].value_counts()
)


print("\nDistribución de nivel_motivacion:")
print(
    df_final["nivel_motivacion"].value_counts()
)


# ============================================================
# 16. VERIFICAR VALORES FALTANTES
# ============================================================

faltantes = df_final.isna().sum().sum()

print("\nValores faltantes:")
print(faltantes)


# ============================================================
# 17. VERIFICAR DUPLICADOS COMPLETOS
# ============================================================

duplicados_completos = df_final.duplicated().sum()

print("\nDuplicados completos:")
print(duplicados_completos)


# ============================================================
# 18. VERIFICAR COPIAS REAL-SINTÉTICO
# ============================================================

firmas_reales_final = set(
    tuple(row)
    for row in df_final[
        df_final["FUENTE_DATO"] == "REAL"
    ][predictoras].to_numpy()
)


firmas_sinteticas_final = [
    tuple(row)
    for row in df_final[
        df_final["FUENTE_DATO"] == "SINTETICO"
    ][predictoras].to_numpy()
]


copias_real_sintetico = sum(
    firma in firmas_reales_final
    for firma in firmas_sinteticas_final
)


duplicados_sinteticos = (
    len(firmas_sinteticas_final)
    - len(set(firmas_sinteticas_final))
)


print("\nCopias exactas real-sintético:")
print(copias_real_sintetico)

print("\nDuplicados entre sintéticos:")
print(duplicados_sinteticos)


# ============================================================
# 19. GUARDAR DATASET DEFINITIVO
# ============================================================

nombre_salida = "DATASET_MACHINE_LEARNING_II.xlsx"

df_final.to_excel(
    nombre_salida,
    index=False
)

print("\n======================================")
print("PROCESO FINALIZADO")
print("======================================")

print(
    f"\nDataset guardado como: {nombre_salida}"
)

Validación de los datos sintéticos y comparación con los datos reales

In [ ]:
# @title
# ============================================================
# PROYECTO MACHINE LEARNING II
# VALIDACIÓN DE LOS DATOS SINTÉTICOS
# Comparación entre registros reales y sintéticos
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. CARGAR EL DATASET DEFINITIVO
# ============================================================

df = pd.read_excel("DATASET_MACHINE_LEARNING_II.xlsx")

print("Dimensiones del dataset:")
print(df.shape)


# ============================================================
# 2. SEPARAR DATOS REALES Y SINTÉTICOS
# ============================================================

df_real = df[
    df["FUENTE_DATO"] == "REAL"
].copy()

df_sintetico = df[
    df["FUENTE_DATO"] == "SINTETICO"
].copy()

print("\nRegistros reales:")
print(len(df_real))

print("\nRegistros sintéticos:")
print(len(df_sintetico))


# ============================================================
# 3. IDENTIFICAR LAS VARIABLES PREDICTORAS
# ============================================================

columnas_excluir = [
    "nivel_motivacion",
    "FUENTE_DATO"
]

predictoras = [
    col for col in df.columns
    if col not in columnas_excluir
]

print("\nNúmero de variables predictoras:")
print(len(predictoras))


# ============================================================
# 4. VALIDACIÓN DE LA DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# ============================================================

print("\n======================================")
print("1. DISTRIBUCIÓN DE NIVEL_MOTIVACION")
print("======================================")

dist_real = (
    df_real["nivel_motivacion"]
    .value_counts(normalize=True)
    .sort_index()
)

dist_sintetico = (
    df_sintetico["nivel_motivacion"]
    .value_counts(normalize=True)
    .sort_index()
)

comparacion_target = pd.DataFrame({
    "REAL": dist_real,
    "SINTETICO": dist_sintetico
})

comparacion_target["DIFERENCIA"] = (
    comparacion_target["SINTETICO"]
    - comparacion_target["REAL"]
).abs()

print(comparacion_target)


# ============================================================
# 5. COMPARACIÓN DE LAS PROPORCIONES DE LAS VARIABLES
# ============================================================

print("\n======================================")
print("2. PROPORCIONES DE LAS VARIABLES")
print("======================================")


diferencias_proporcion = []

for col in predictoras:

    proporcion_real = df_real[col].mean()
    proporcion_sintetico = df_sintetico[col].mean()

    diferencia = abs(
        proporcion_real
        - proporcion_sintetico
    )

    diferencias_proporcion.append(diferencia)


diferencias_proporcion = np.array(
    diferencias_proporcion
)

print(
    "Diferencia promedio:",
    diferencias_proporcion.mean()
)

print(
    "Diferencia máxima:",
    diferencias_proporcion.max()
)

print(
    "Variables con diferencia <= 0.05:",
    np.sum(
        diferencias_proporcion <= 0.05
    ),
    "de",
    len(predictoras)
)

print(
    "Variables con diferencia <= 0.10:",
    np.sum(
        diferencias_proporcion <= 0.10
    ),
    "de",
    len(predictoras)
)


# ============================================================
# 6. IDENTIFICAR LAS VARIABLES CON MAYOR DIFERENCIA
# ============================================================

tabla_proporciones = pd.DataFrame({
    "VARIABLE": predictoras,
    "DIFERENCIA": diferencias_proporcion
})

tabla_proporciones = (
    tabla_proporciones
    .sort_values(
        "DIFERENCIA",
        ascending=False
    )
)

print("\n10 variables con mayor diferencia:")
print(
    tabla_proporciones.head(10)
)


# ============================================================
# 7. COMPARACIÓN DE LA ESTRUCTURA DE CORRELACIONES
# ============================================================

print("\n======================================")
print("3. ESTRUCTURA DE CORRELACIONES")
print("======================================")


# Matrices de correlación
corr_real = df_real[
    predictoras
].corr()

corr_sintetico = df_sintetico[
    predictoras
].corr()


# Diferencia absoluta entre matrices
diferencia_corr = (
    corr_real
    - corr_sintetico
).abs()


# Tomar únicamente el triángulo superior
# para no contar dos veces cada par
mask = np.triu(
    np.ones(
        diferencia_corr.shape,
        dtype=bool
    ),
    k=1
)


valores_corr = diferencia_corr[
    mask
]


# Eliminar valores NaN
valores_corr = valores_corr[
    ~np.isnan(valores_corr)
]


print(
    "Diferencia media entre correlaciones:",
    valores_corr.mean()
)

print(
    "Diferencia máxima entre correlaciones:",
    valores_corr.max()
)

print(
    "Pares con diferencia <= 0.10:",
    np.mean(
        valores_corr <= 0.10
    )
)


# ============================================================
# 8. RELACIÓN DE LAS VARIABLES CON NIVEL_MOTIVACION
# ============================================================

print("\n======================================")
print("4. RELACIÓN CON NIVEL_MOTIVACION")
print("======================================")


# Convertir temporalmente las categorías
# de nivel_motivacion a códigos numéricos
categorias = sorted(
    df["nivel_motivacion"].unique()
)

mapa_target = {
    categoria: i
    for i, categoria in enumerate(categorias)
}


target_real = (
    df_real["nivel_motivacion"]
    .map(mapa_target)
)

target_sintetico = (
    df_sintetico["nivel_motivacion"]
    .map(mapa_target)
)


# Calcular correlación de cada variable
# con la variable objetivo
relacion_real = (
    df_real[predictoras]
    .corrwith(target_real)
)

relacion_sintetica = (
    df_sintetico[predictoras]
    .corrwith(target_sintetico)
)


# Diferencia absoluta
diferencia_target = (
    relacion_real
    - relacion_sintetica
).abs()


print(
    "Diferencia promedio:",
    diferencia_target.mean()
)

print(
    "Diferencia máxima:",
    diferencia_target.max()
)


# Variables con mayor diferencia
tabla_target = pd.DataFrame({
    "VARIABLE": predictoras,
    "DIFERENCIA": diferencia_target
})

tabla_target = (
    tabla_target
    .sort_values(
        "DIFERENCIA",
        ascending=False
    )
)

print("\n10 variables con mayor diferencia:")
print(
    tabla_target.head(10)
)


# ============================================================
# 9. COMPROBAR COPIAS EXACTAS ENTRE REAL Y SINTÉTICO
# ============================================================

print("\n======================================")
print("5. COPIAS EXACTAS")
print("======================================")


firmas_reales = set(
    tuple(row)
    for row in df_real[
        predictoras
    ].to_numpy()
)


firmas_sinteticas = [
    tuple(row)
    for row in df_sintetico[
        predictoras
    ].to_numpy()
]


copias_real_sintetico = sum(
    firma in firmas_reales
    for firma in firmas_sinteticas
)


print(
    "Copias exactas real-sintético:",
    copias_real_sintetico
)


# ============================================================
# 10. COMPROBAR DUPLICADOS ENTRE SINTÉTICOS
# ============================================================

duplicados_sinteticos = (
    len(firmas_sinteticas)
    - len(set(firmas_sinteticas))
)


print(
    "Duplicados entre sintéticos:",
    duplicados_sinteticos
)


# ============================================================
# 11. COMPROBAR VALORES FALTANTES
# ============================================================

print("\n======================================")
print("6. VALORES FALTANTES")
print("======================================")


faltantes_real = (
    df_real[predictoras]
    .isna()
    .sum()
    .sum()
)

faltantes_sintetico = (
    df_sintetico[predictoras]
    .isna()
    .sum()
    .sum()
)


print(
    "Valores faltantes en reales:",
    faltantes_real
)

print(
    "Valores faltantes en sintéticos:",
    faltantes_sintetico
)


# ============================================================
# 12. RESUMEN FINAL DE LA VALIDACIÓN
# ============================================================

print("\n======================================")
print("RESUMEN DE VALIDACIÓN")
print("======================================")


print(
    "\nDiferencia promedio de proporciones:",
    diferencias_proporcion.mean()
)

print(
    "Diferencia media de correlaciones:",
    valores_corr.mean()
)

print(
    "Proporción de pares con diferencia de correlación <= 0.10:",
    np.mean(valores_corr <= 0.10)
)

print(
    "Diferencia promedio de relación con nivel_motivacion:",
    diferencia_target.mean()
)

print(
    "Copias exactas real-sintético:",
    copias_real_sintetico
)

print(
    "Duplicados entre sintéticos:",
    duplicados_sinteticos
)

print(
    "Valores faltantes reales:",
    faltantes_real
)

print(
    "Valores faltantes sintéticos:",
    faltantes_sintetico
)

print("\n======================================")
print("VALIDACIÓN FINALIZADA")
print("======================================")

SEGUNDO PUNTO

Código del flujo reproducible sin fuga de datos

In [ ]:
# @title
# ============================================================
# PROYECTO MACHINE LEARNING II
# FLUJO REPRODUCIBLE SIN FUGA DE DATOS
# ============================================================

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. CARGAR EL DATASET DEFINITIVO
# ============================================================

df = pd.read_excel(
    "DATASET_MACHINE_LEARNING_II.xlsx"
)


# ============================================================
# 2. CONVERTIR EL PROBLEMA EN CLASIFICACIÓN BINARIA
# ============================================================

df["riesgo"] = df["nivel_motivacion"].map({
    "Deserción riesgo muy bajo": 0,
    "Deserción riesgo leve": 0,
    "Deserción riesgo medio": 1,
    "Deserción riesgo alto": 1,
    "Deserción casi segura": 1
})


# ============================================================
# 3. DEFINIR VARIABLES PREDICTORAS Y VARIABLE OBJETIVO
# ============================================================

X = df.drop(
    columns=[
        "nivel_motivacion",
        "riesgo",
        "FUENTE_DATO"
    ]
)

y = df["riesgo"]


# ============================================================
# 4. DIVIDIR LOS DATOS EN ENTRENAMIENTO Y PRUEBA
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("Datos de entrenamiento:")
print(X_train.shape)

print("\nDatos de prueba:")
print(X_test.shape)


# ============================================================
# 5. CREAR EL PIPELINE
# ============================================================

pipeline_svm = Pipeline([

    (
        "escalamiento",
        StandardScaler()
    ),

    (
        "modelo",
        SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        )
    )
])


# ============================================================
# 6. ENTRENAR EL PIPELINE
# ============================================================

pipeline_svm.fit(
    X_train,
    y_train
)


# ============================================================
# 7. REALIZAR PREDICCIONES
# ============================================================

y_pred = pipeline_svm.predict(
    X_test
)


# ============================================================
# 8. MATRIZ DE CONFUSIÓN
# ============================================================

print("\n======================================")
print("MATRIZ DE CONFUSIÓN")
print("======================================")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


# ============================================================
# 9. REPORTE DE CLASIFICACIÓN
# ============================================================

print("\n======================================")
print("REPORTE DE CLASIFICACIÓN")
print("======================================")

print(
    classification_report(
        y_test,
        y_pred
    )
)


# ============================================================
# 10. EXACTITUD
# ============================================================

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\n======================================")
print("EXACTITUD")
print("======================================")

print(
    f"Accuracy: {accuracy:.2%}"
)


# ============================================================
# 11. RECALL DE LA CLASE DE RIESGO
# ============================================================

from sklearn.metrics import recall_score

recall_riesgo = recall_score(
    y_test,
    y_pred,
    pos_label=1
)

print("\n======================================")
print("RECALL DE ESTUDIANTES EN RIESGO")
print("======================================")

print(
    f"Recall: {recall_riesgo:.2%}"
)

SEGUNDO Y TERCER PUNTO

Instalación e importaciones

In [ ]:
%pip install -q shap

In [ ]:
import os, random, time, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             fbeta_score, roc_auc_score, confusion_matrix, make_scorer)
from sklearn.model_selection import (StratifiedKFold, StratifiedGroupKFold, GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.makedirs("resultados", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25})
print("Librerías cargadas. SEED =", SEED)

1. Carga de datos y variable objetivo

In [ ]:
RUTA = "DATASET_MACHINE_LEARNING_II.xlsx"

if not os.path.exists(RUTA):
    from google.colab import files
    files.upload()

df = pd.read_excel(RUTA)

EN_RIESGO = [
    "Deserción riesgo medio",
    "Deserción riesgo alto",
    "Deserción casi segura"
]

y = (
    df["nivel_motivacion"]
    .isin(EN_RIESGO)
    .astype(int)
    .rename("riesgo")
)

fuente = df["FUENTE_DATO"].copy()

X = df.drop(
    columns=[
        "nivel_motivacion",
        "FUENTE_DATO"
    ]
).astype(np.float32)


print("Forma de X:", X.shape)
print(
    "Valores faltantes:",
    int(X.isna().sum().sum())
)

print("\nDistribución de la variable objetivo:")
print(
    pd.concat(
        [
            y.value_counts().rename("n"),
            y.value_counts(normalize=True)
             .round(3)
             .rename("prop")
        ],
        axis=1
    )
)

print("\nFuente de los datos:")
print(
    fuente.value_counts()
)

print("\nCategorías originales por clase:")
print(
    pd.crosstab(
        df["nivel_motivacion"],
        y
    )
)

2. Auditoría de duplicados

In [ ]:
grupos = pd.util.hash_pandas_object(X, index=False).values

aud = pd.DataFrame({
    "registros": [len(X), int((fuente=="REAL").sum()), int((fuente=="SINTETICO").sum())],
    "perfiles_unicos": [len(np.unique(grupos)),
                        len(np.unique(grupos[(fuente=="REAL").values])),
                        len(np.unique(grupos[(fuente=="SINTETICO").values]))],
}, index=["Total", "REAL", "SINTETICO"])
aud["registros_repetidos"] = aud["registros"] - aud["perfiles_unicos"]
print(aud)

etq = pd.DataFrame({"g": grupos, "y": y.values}).groupby("g")["y"].nunique()
print("\nPerfiles con etiquetas contradictorias:", int((etq > 1).sum()))
print("Máximo de copias de un mismo perfil real:", int(pd.Series(grupos[(fuente=='REAL').values]).value_counts().max()))

3. Partición hold-out 80/20 sin fuga

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
idx_tr, idx_te = next(sgkf.split(X, y, groups=grupos))

X_tr, X_te = X.iloc[idx_tr], X.iloc[idx_te]
y_tr, y_te = y.iloc[idx_tr], y.iloc[idx_te]
g_tr, g_te = grupos[idx_tr], grupos[idx_te]
f_te = fuente.iloc[idx_te]

print(f"Entrenamiento: {len(X_tr)} | Prueba: {len(X_te)}")
print("Perfiles compartidos entre entrenamiento y prueba:", len(set(g_tr) & set(g_te)))
print("Proporción en riesgo -> train: %.3f | test: %.3f" % (y_tr.mean(), y_te.mean()))
print("Sintéticos en prueba:", int((f_te=="SINTETICO").sum()))

3.1 Demostración: cuánto infla las métricas la validación sin agrupar

In [ ]:
def cv_rapida(cv, usar_grupos):
    res = []

    splits = (
        cv.split(X, y, groups=grupos)
        if usar_grupos
        else cv.split(X, y)
    )

    for a, b in splits:
        m = RandomForestClassifier(
            n_estimators=200,
            random_state=SEED,
            n_jobs=-1
        ).fit(X.iloc[a], y.iloc[a])

        p = m.predict(X.iloc[b])

        res.append((
            accuracy_score(y.iloc[b], p),
            recall_score(y.iloc[b], p),
            f1_score(y.iloc[b], p)
        ))

    return np.array(res).mean(axis=0)


r_sin = cv_rapida(
    StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    ),
    False
)

r_con = cv_rapida(
    StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    ),
    True
)

fuga = pd.DataFrame(
    [r_sin, r_con],
    index=[
        "StratifiedKFold (sin agrupar)",
        "StratifiedGroupKFold (agrupado)"
    ],
    columns=[
        "Accuracy",
        "Recall riesgo",
        "F1 riesgo"
    ]
)

print(fuga.round(4))

fuga.to_csv(
    "resultados/demostracion_fuga.csv"
)

4. Métricas y criterio de decisión (adaptación del artículo)

In [ ]:
f2_scorer = make_scorer(fbeta_score, beta=2)

def puntaje(modelo, X_):
    return modelo.predict_proba(X_)[:, 1] if hasattr(modelo, "predict_proba") else modelo.decision_function(X_)

def metricas(modelo, X_, y_):
    p = modelo.predict(X_)
    return {
        "accuracy":     accuracy_score(y_, p),
        "precision":    precision_score(y_, p, zero_division=0),
        "recall":       recall_score(y_, p),
        "especificidad": recall_score(y_, p, pos_label=0),
        "f1":           f1_score(y_, p),
        "f2":           fbeta_score(y_, p, beta=2),
        "roc_auc":      roc_auc_score(y_, puntaje(modelo, X_)),
    }
COLS = ["accuracy", "precision", "recall", "especificidad", "f1", "f2", "roc_auc"]

5. Modelos, línea base y espacios de búsqueda

In [ ]:
from sklearn.model_selection import cross_validate
import matplotlib.pyplot as plt
import pandas as pd

k_valores = [10, 30, 60, 100, 150, X_tr.shape[1]]
resultados_k = []

print("Evaluando el rendimiento según la cantidad de variables (k)...")

for k in k_valores:

    pipe_prueba = Pipeline([
        ("sel", SelectKBest(f_classif, k=k)),
        ("m", RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        ))
    ])

    cv_res = cross_validate(
        pipe_prueba,
        X_tr,
        y_tr,
        groups=g_tr,
        cv=StratifiedGroupKFold(
            n_splits=3,
            shuffle=True,
            random_state=SEED
        ),
        scoring={
            "recall": "recall",
            "f2": f2_scorer
        }
    )

    resultados_k.append({
        "k": k,
        "Recall": cv_res["test_recall"].mean(),
        "F2": cv_res["test_f2"].mean()
    })

    print(
        f"Probando k = {k:3d} | "
        f"Recall: {cv_res['test_recall'].mean():.3f} | "
        f"F2: {cv_res['test_f2'].mean():.3f}"
    )

df_k = pd.DataFrame(resultados_k)

print("\nResultados:")
print(df_k.round(4))

plt.figure(figsize=(8, 4.5))
plt.plot(
    df_k["k"],
    df_k["Recall"],
    marker="o",
    linewidth=2,
    label="Recall (Sensibilidad)"
)

plt.plot(
    df_k["k"],
    df_k["F2"],
    marker="s",
    linewidth=2,
    label="F2"
)

plt.axvline(
    x=30,
    linestyle="--",
    label="k = 30"
)

plt.title("Análisis de sensibilidad según número de variables")
plt.xlabel("Número de variables seleccionadas (k)")
plt.ylabel("Rendimiento")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def pipe(nombre):

    if nombre == "Baseline mayoría":
        return DummyClassifier(strategy="most_frequent")

    if nombre == "Baseline Regresión logística":
        return Pipeline([
            ("esc", StandardScaler()),
            ("m", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=SEED
            ))
        ])

    if nombre == "SVM (RBF)":
        return Pipeline([
            ("esc", StandardScaler()),
            ("m", SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=True,
                random_state=SEED
            ))
        ])

    if nombre == "Random Forest":
        return Pipeline([
            ("m", RandomForestClassifier(
                class_weight="balanced",
                random_state=SEED,
                n_jobs=1
            ))
        ])

    if nombre == "Boosting (HistGB)":
        return Pipeline([
            ("m", HistGradientBoostingClassifier(
                class_weight="balanced",
                random_state=SEED
            ))
        ])

    if nombre == "RF + selección (top-30)":
        return Pipeline([
            ("sel", SelectKBest(f_classif, k=30)),
            ("m", RandomForestClassifier(
                class_weight="balanced",
                random_state=SEED,
                n_jobs=1
            ))
        ])


REJILLAS = {
    "Baseline mayoría": None,

    "Baseline Regresión logística": None,

    "SVM (RBF)": {
        "m__C": [1, 10],
        "m__gamma": ["scale", 0.001]
    },

    "Random Forest": {
        "m__n_estimators": [200],
        "m__max_depth": [10, None],
        "m__min_samples_split": [2, 10]
    },

    "Boosting (HistGB)": {
        "m__learning_rate": [0.05, 0.1],
        "m__max_depth": [3, 6],
        "m__max_iter": [200]
    },

    "RF + selección (top-30)": {
        "m__n_estimators": [200],
        "m__max_depth": [10, None]
    },
}


MODELOS = list(REJILLAS)

AVANZADOS = [
    "SVM (RBF)",
    "Random Forest",
    "Boosting (HistGB)",
    "RF + selección (top-30)"
]


def ajustar(nombre, X_, y_, g_, n_splits_interno):
    """
    Ajusta el modelo usando validación cruzada agrupada
    y F2 como criterio de selección.
    """

    base = pipe(nombre)

    if REJILLAS[nombre] is None:
        return clone(base).fit(X_, y_), None

    gs = GridSearchCV(
        estimator=base,
        param_grid=REJILLAS[nombre],
        scoring=f2_scorer,
        cv=StratifiedGroupKFold(
            n_splits=n_splits_interno,
            shuffle=True,
            random_state=SEED
        ),
        n_jobs=-1,
        refit=True
    )

    gs.fit(
        X_,
        y_,
        groups=g_
    )

    return gs.best_estimator_, gs.best_params_


print("Modelos a comparar:", MODELOS)

Número exacto de gamma en SVM

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X_scaled = StandardScaler().fit_transform(X)

svm_temp = SVC(
    kernel="rbf",
    gamma="scale"
)

svm_temp.fit(X_scaled, y)

print(
    "Gamma calculado en esta prueba exploratoria:",
    svm_temp._gamma
)

Métricas de cada modelo

In [ ]:
tabla_resultados = []

for nombre in MODELOS:

    modelo_actual, mejores_params = ajustar(
        nombre,
        X_tr,
        y_tr,
        g_tr,
        n_splits_interno=3
    )

    metricas_test = metricas(
        modelo_actual,
        X_te,
        y_te
    )

    tabla_resultados.append({
        "Modelo": nombre,
        "Accuracy": round(metricas_test["accuracy"], 4),
        "Precision": round(metricas_test["precision"], 4),
        "Recall (Clase Riesgo)": round(metricas_test["recall"], 4),
        "Especificidad": round(metricas_test["especificidad"], 4),
        "F1-score": round(metricas_test["f1"], 4),
        "F2-score": round(metricas_test["f2"], 4),
        "ROC-AUC": round(metricas_test["roc_auc"], 4),
        "Mejores parámetros": mejores_params
    })

df_final_metrics = pd.DataFrame(tabla_resultados)

display(df_final_metrics)

6. Validación cruzada anidada

In [ ]:
cv_ext = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
filas, oof = [], {m: np.zeros(len(X_tr), dtype=int) for m in MODELOS}
mejores_params = {m: [] for m in MODELOS}

t0 = time.time()
for nombre in MODELOS:
    for k, (a, b) in enumerate(cv_ext.split(X_tr, y_tr, groups=g_tr), 1):
        modelo, bp = ajustar(nombre, X_tr.iloc[a], y_tr.iloc[a], g_tr[a], 3)
        r = metricas(modelo, X_tr.iloc[b], y_tr.iloc[b])
        r.update(modelo=nombre, pliegue=k)
        filas.append(r)
        oof[nombre][b] = modelo.predict(X_tr.iloc[b])
        mejores_params[nombre].append(bp)
    print(f"{nombre:<32} listo  ({time.time()-t0:5.0f} s acumulados)")

cv_df = pd.DataFrame(filas)
cv_df.to_csv("resultados/cv_anidada_por_pliegue.csv", index=False)

In [ ]:
resumen = cv_df.groupby("modelo")[COLS].agg(["mean", "std"]).loc[MODELOS]
tabla = pd.DataFrame({c: resumen[c]["mean"].round(3).astype(str) + " ± " + resumen[c]["std"].round(3).astype(str)
                      for c in COLS})
print("Validación cruzada anidada (media ± desviación estándar, 5 pliegues):")
print(tabla.to_string())
resumen.to_csv("resultados/cv_anidada_resumen.csv")
tabla.to_csv("resultados/cv_anidada_tabla.csv")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=False)

for a_, met, tit in zip(
    ax,
    ["recall", "f1"],
    ["Recall clase en riesgo", "F1 clase en riesgo"]
):
    datos = [
        cv_df.loc[cv_df["modelo"] == m, met].values
        for m in MODELOS
    ]

    a_.boxplot(
        datos,
        tick_labels=[
            m.replace(" ", "\n", 1)
            for m in MODELOS
        ],
        showmeans=True
    )

    a_.set_title(
        tit + " por pliegue (CV anidada)"
    )
    a_.set_ylim(0.5, 1.02)
    a_.tick_params(axis="x", labelsize=8)
    a_.set_ylabel("Puntuación")

plt.tight_layout()

plt.savefig(
    "resultados/fig1_estabilidad_cv.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

6.1 Hiperparámetros elegidos en cada pliegue

In [ ]:
for m in AVANZADOS:
    print(m)
    for k, bp in enumerate(mejores_params[m], 1):
        print(f"   pliegue {k}: {bp}")

7. Selección del modelo y evaluación final en el hold-out

In [ ]:
sel = (
    cv_df[cv_df["modelo"].isin(AVANZADOS)]
    .groupby("modelo")["f2"]
    .agg(["mean", "std"])
)

print(sel.round(4))

In [ ]:
finales, params_finales, preds_te = {}, {}, {}

fila_te = []

for nombre in MODELOS:

    modelo, bp = ajustar(
        nombre,
        X_tr,
        y_tr,
        g_tr,
        n_splits_interno=5
    )

    finales[nombre] = modelo
    params_finales[nombre] = bp

    preds_te[nombre] = modelo.predict(X_te)

    r = metricas(
        modelo,
        X_te,
        y_te
    )
    r["modelo"] = nombre
    fila_te.append(r)

holdout = (
    pd.DataFrame(fila_te)
    .set_index("modelo")[COLS]
    .loc[MODELOS]
)

print(
    f"Evaluación en hold-out "
    f"({len(X_te)} registros, una sola vez):"
)

print(holdout.round(3).to_string())

holdout.to_csv(
    "resultados/holdout_metricas.csv"
)

print("\nHiperparámetros finales:")

for m in AVANZADOS:
    print("  ", m, params_finales[m])

8. Matriz de confusión y análisis de errores

In [ ]:
cm_te = confusion_matrix(y_te, preds_te[MEJOR])
cm_oof = confusion_matrix(y_tr, oof[MEJOR])

tn, fp, fn, tp = cm_te.ravel()

print(
    f"HOLD-OUT ({MEJOR}) -> "
    f"VN={tn}  FP={fp}  FN={fn}  VP={tp}"
)

print(
    f"  En riesgo reales: {tp+fn} | "
    f"detectados: {tp} ({tp/(tp+fn):.1%}) | "
    f"no detectados: {fn} ({fn/(tp+fn):.1%})"
)

print(
    f"  Alertas innecesarias (FP): {fp} de {tn+fp} "
    f"estudiantes sin riesgo ({fp/(tn+fp):.1%})"
)

fig, ax = plt.subplots(1, 2, figsize=(9.5, 4))

for a_, cm, tit in zip(
    ax,
    [cm_oof, cm_te],
    [
        "CV anidada (fuera de muestra, train)",
        "Hold-out (prueba)"
    ]
):

    a_.imshow(cm, cmap="Blues")
    a_.set_title(tit, fontsize=10)
    a_.grid(False)

    a_.set_xticks([0, 1])
    a_.set_yticks([0, 1])
    a_.set_xticklabels(["No riesgo", "En riesgo"])
    a_.set_yticklabels(["No riesgo", "En riesgo"])

    a_.set_xlabel("Predicción")
    a_.set_ylabel("Real")

    for i in range(2):
        for j in range(2):
            a_.text(
                j,
                i,
                cm[i, j],
                ha="center",
                va="center",
                fontsize=14,
                color="white" if cm[i, j] > cm.max() / 2 else "black"
            )

plt.suptitle(
    f"Matriz de confusión — {MEJOR}"
)

plt.tight_layout()

plt.savefig(
    "resultados/fig2_matriz_confusion.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
cat_te = df["nivel_motivacion"].iloc[idx_te].values
err = pd.DataFrame({"cat": cat_te, "real": y_te.values, "pred": preds_te[MEJOR]})
err["error"] = (err.real != err.pred).astype(int)
por_cat = err.groupby("cat").agg(n=("error", "size"), errores=("error", "sum"))
por_cat["tasa_error"] = (por_cat.errores / por_cat.n).round(3)
orden = ["Deserción riesgo muy bajo", "Deserción riesgo leve", "Deserción riesgo medio",
         "Deserción riesgo alto", "Deserción casi segura"]
por_cat = por_cat.loc[orden]
print("Errores por categoría original (hold-out):"); print(por_cat)
por_cat.to_csv("resultados/errores_por_categoria.csv")

8.1 Robustez frente a los datos sintéticos

In [ ]:
rob = []
for etiqueta, mask in [("REAL", (f_te == "REAL").values), ("SINTETICO", (f_te == "SINTETICO").values)]:
    yy, pp = y_te.values[mask], preds_te[MEJOR][mask]
    rob.append({"fuente": etiqueta, "n": int(mask.sum()),
                "accuracy": accuracy_score(yy, pp), "recall": recall_score(yy, pp),
                "precision": precision_score(yy, pp), "f1": f1_score(yy, pp)})
rob = pd.DataFrame(rob).set_index("fuente")
print(rob.round(3)); rob.to_csv("resultados/robustez_real_vs_sintetico.csv")

9. Interpretación preliminar del modelo seleccionado

In [ ]:
def variable_de(col):

    return col.rsplit("_", 1)[0] if "_" in col else col


mapa = pd.Series({
    c: variable_de(c)
    for c in X.columns
})

bloques = {
    v: list(mapa[mapa == v].index)
    for v in mapa.unique()
}

print(
    "Variables originales (grupos de columnas):",
    len(bloques)
)


def importancia_agrupada(
    modelo,
    X_,
    y_,
    n_rep=5,
    seed=SEED
):
    rng = np.random.RandomState(seed)

    base = fbeta_score(
        y_,
        modelo.predict(X_),
        beta=2
    )

    out = {}

    for v, cols in bloques.items():

        caidas = []

        for _ in range(n_rep):
            Xp = X_.copy()


            Xp[cols] = Xp[cols].iloc[
                rng.permutation(len(Xp))
            ].values

            caidas.append(
                base - fbeta_score(
                    y_,
                    modelo.predict(Xp),
                    beta=2
                )
            )

        out[v] = (
            np.mean(caidas),
            np.std(caidas)
        )

    return (
        pd.DataFrame(
            out,
            index=["caida_F2", "desv"]
        )
        .T
        .sort_values(
            "caida_F2",
            ascending=False
        )
    )


imp = importancia_agrupada(
    finales[MEJOR],
    X_te,
    y_te
)

imp.to_csv(
    "resultados/importancia_permutacion_agrupada.csv"
)

print(
    imp.head(15).round(4)
)


top = imp.head(15).iloc[::-1]

plt.figure(figsize=(7.5, 5))

plt.barh(
    top.index,
    top["caida_F2"],
    xerr=top["desv"]
)

plt.xlabel(
    "Caída del F2 al permutar el bloque (hold-out)"
)

plt.title(
    f"Importancia predictiva agrupada — {MEJOR}"
)

plt.tight_layout()

plt.savefig(
    "resultados/fig3_importancia_permutacion.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

9.1 SHAP

In [ ]:
try:
    import shap

    modelo = finales[MEJOR]

    if hasattr(modelo, "predict_proba"):
        f = lambda A: modelo.predict_proba(
            pd.DataFrame(A, columns=X.columns)
        )[:, 1]
    else:
        f = lambda A: modelo.decision_function(
            pd.DataFrame(A, columns=X.columns)
        )

    fondo = shap.sample(
        X_tr,
        50,
        random_state=SEED
    ).values

    muestra = X_te.sample(
        150,
        random_state=SEED
    )

    explicador = shap.Explainer(
        f,
        fondo,
        algorithm="permutation"
    )

    sv = explicador(
        muestra.values,
        max_evals=2 * X.shape[1] + 1
    )

    abs_media = pd.Series(
        np.abs(sv.values).mean(axis=0),
        index=X.columns
    )

    shap_var = (
        abs_media
        .groupby(mapa)
        .sum()
        .sort_values(ascending=False)
    )

    shap_var.to_csv(
        "resultados/shap_por_variable.csv"
    )

    print(
        shap_var.head(15).round(4)
    )

    top = shap_var.head(15).iloc[::-1]

    plt.figure(figsize=(7.5, 5))

    plt.barh(
        top.index,
        top.values
    )

    plt.xlabel(
        "|SHAP| medio (suma de columnas de la variable)"
    )

    plt.title(
        f"SHAP agregado por variable — {MEJOR}"
    )

    plt.tight_layout()

    plt.savefig(
        "resultados/fig4_shap_por_variable.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

    comun = (
        set(imp.head(10).index)
        & set(shap_var.head(10).index)
    )

    print(
        "Variables en el top-10 de ambos métodos:",
        sorted(comun)
    )

except Exception as e:
    print(
        "SHAP no se pudo ejecutar:",
        type(e).__name__,
        e
    )

10. Resumen consolidado

In [ ]:
print("=" * 70)

print(
    "TABLA 1. Validación cruzada anidada "
    "(media ± desv., 5 pliegues, conjunto de entrenamiento)"
)

print(
    tabla[
        ["accuracy", "recall", "precision", "f1", "roc_auc"]
    ].to_string()
)

print("\nTABLA 2. Hold-out (conjunto de prueba)")

print(
    holdout[
        [
            "accuracy",
            "recall",
            "precision",
            "especificidad",
            "f1",
            "roc_auc"
        ]
    ].round(3).to_string()
)

print("\nModelo seleccionado:", MEJOR)

print("\nDemostración de fuga (RF por defecto):")
print(fuga.round(4).to_string())

resumen_json = {
    "modelo_seleccionado": MEJOR,
    "n_train": int(len(X_tr)),
    "n_test": int(len(X_te)),
    "hold_out_confusion": {
        "VN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "VP": int(tp)
    },
    "params_finales": {
        k: v
        for k, v in params_finales.items()
        if v
    }
}

with open(
    "resultados/resumen.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        resumen_json,
        f,
        ensure_ascii=False,
        indent=2,
        default=str
    )